# Walk forward validation et tracking metrics

Ce notebook sert de benchmark de performance pour les trois modèles entrainés. Les métriques de comparaison sont :
- RMSE ;
- MAE ;
- SMAPE ;
- Skill Score over persistance


Il sera aussi comparé des métriques sur tous les horizons. Enfin, deux périmètres importants seront extraits pour comparer la performance des modèles sur ces périmètres :
- Temps nuageux (cloud_cover > 50) ;
- Par saison ;

In [13]:
# Import librairies
# Registry
import mlflow
from mlflow.pyfunc import PythonModel
from dotenv import dotenv_values

# ML / DL
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split

# Visualisation
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

#Logs, typing, libraries standards
import traceback
import logging 
import os
from math import inf
import joblib
import warnings
from typing import Dict, List, Optional, Any, Tuple, Set
from dataclasses import dataclass
import pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore")


# Config basique logging
PROJECT_ROOT = Path().resolve().parent
logging.basicConfig(filename=PROJECT_ROOT / "logs/data_training.log", level=logging.INFO, datefmt="%Y-%m-%d-%H-%M-%s")
logging.getLogger('mlflow').setLevel(logging.ERROR)

In [14]:
def init_mlflow(config: dict, experiment_name: str) -> None:
    """
    MLFlow initializing
    """
    os.environ["MLFLOW_TRACKING_USERNAME"] = config["MLFLOW_TRACKING_USERNAME"]
    os.environ["MLFLOW_TRACKING_PASSWORD"] = config["MLFLOW_TRACKING_PASSWORD"]
    mlflow.set_tracking_uri(config["MLFLOW_TRACKING_URI"])
    mlflow.set_experiment(experiment_name=experiment_name)

In [16]:
# Data core
df = pd.read_csv(PROJECT_ROOT / "data/processed/training_dataset.csv", index_col=0)
df.index = pd.to_datetime(df.index, utc=True).tz_convert("Europe/Paris")
meteo_features = joblib.load(PROJECT_ROOT / "data/processed/features/meteo_features.plk")

# Features
X = df.drop(columns="solar_mw")
y = df["solar_mw"]

# Séparation train test split
X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y, 
                                                    test_size=0.15,
                                                    shuffle=False, 
                                                    random_state=42)

# Configuratio secrets
config = dotenv_values(PROJECT_ROOT / ".env")
init_mlflow(config=config, experiment_name="LGBM_Direct_Output")

In [ ]:
model_uri = config["MODEL_URI"]
model = mlflow.pyfunc.load_model(model_uri)

## Walk forward validation (in progress)

En premier lieu, nous mettons en place une fonction de walk forward simple avec generator.

In [143]:
def rmse(y_true: pd.Series, y_pred: pd.Series) -> Any:
    """Return rmse of y_true and y_prec"""
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [144]:
class PersistenceBaseline:
    """Baseline lag-24h : demain = aujourd'hui."""
    
    def predict(self, window: pd.Series) -> pd.DataFrame:
        last_24 = window.iloc[-24:].values  
        
        horizon_cols = [f"H+{h}" for h in range(1, 25)]
        return pd.DataFrame(
            [last_24],
            columns=horizon_cols
        )

In [ ]:
def walk_forward_eval(
    input_dataset: pd.DataFrame,
    true_values: pd.Series,
    wrapped_model,
    model_input_length: int,
    minimal_window: int,
    stride: int) -> Tuple[np.ndarray, np.ndarray]:

    # Init 
    observed_values, predictions = [], []

    for step in tqdm(range(minimal_window, len(true_values) - 24, stride)):
        window = input_dataset.iloc[step - model_input_length:step]  
        
        # Method among 
        if isinstance(wrapped_model, PersistenceBaseline):
            y_hat  = wrapped_model.predict(true_values.iloc[:step]).values.flatten()  
        else:
            y_hat  = wrapped_model.predict(window).values.flatten()       
        
        # Predictions
        y_true = true_values.iloc[step:step + 24].values             

        observed_values.append(y_true)
        predictions.append(y_hat)

    return np.array(observed_values), np.array(predictions)  

In [ ]:
def compute_metrics(
    observed_values: np.ndarray,  
    predictions: np.ndarray,      
    baseline_predictions: np.ndarray,
) -> Dict[str, Any]:
    
    err      = observed_values - predictions
    err_base = observed_values - baseline_predictions

    # Metrics
    rmse_global  = np.sqrt(np.mean(err**2))
    mae_global   = np.mean(np.abs(err))
    rmse_base    = np.sqrt(np.mean(err_base**2))
    skill_score  = 1 - rmse_global / rmse_base

    # Par horizon
    horizon_idx          = [f"H+{h}" for h in range(1, 25)]
    rmse_per_horizon     = pd.Series(np.sqrt(np.mean(err**2,      axis=0)), index=horizon_idx)
    mae_per_horizon      = pd.Series(np.mean(np.abs(err),         axis=0), index=horizon_idx)
    ss_per_horizon       = pd.Series(
        1 - np.sqrt(np.mean(err**2, axis=0)) / np.sqrt(np.mean(err_base**2, axis=0)),
        index=horizon_idx
    )

    return {
        "RMSE":             rmse_global,
        "MAE":              mae_global,
        "SkillScore":       skill_score,
        "RMSE_per_horizon": rmse_per_horizon,
        "MAE_per_horizon":  mae_per_horizon,
        "SS_per_horizon":   ss_per_horizon,
    }

In [ ]:
baseline = PersistenceBaseline()
obs, preds_base = walk_forward_eval(
    input_dataset=X_test, true_values=y_test, 
    wrapped_model=baseline,
    model_input_length=24,
    minimal_window=48,
    stride=24
)

obs, preds_lgbm = walk_forward_eval(
    input_dataset=X_test,
    true_values=y_test,
    wrapped_model= model,
    model_input_length=1,
    minimal_window=48,
    stride=24
)

metrics_lgbm = compute_metrics(obs, preds_lgbm, preds_base)

print(metrics_lgbm["SkillScore"])
print(metrics_lgbm["SS_per_horizon"])

100%|██████████| 123/123 [00:40<00:00,  3.04it/s]

0.15862594229385119
H+1         -inf
H+2         -inf
H+3         -inf
H+4         -inf
H+5         -inf
H+6         -inf
H+7         -inf
H+8    -1.897349
H+9    -0.550401
H+10   -0.035478
H+11    0.094822
H+12    0.080610
H+13    0.189900
H+14    0.192661
H+15    0.222185
H+16    0.215038
H+17    0.206389
H+18    0.133543
H+19    0.030971
H+20   -0.307440
H+21   -0.899564
H+22   -2.283640
H+23   -3.054794
H+24   -0.675871
dtype: float64
